---
title: "Untitled"
output: html_document
date: "2025-10-06"
---



In [ ]:
knitr::opts_chunk$set(echo = TRUE)




## R libraries.



In [1]:
library(ggplot2)
library(ggrepel)
library(ggpubr)
library(stringr)
library(MASS)
library(RColorBrewer)

library(viridis)
library(ggpointdensity)
library(dplyr)
library(data.table)
library(readxl)

theme_set(theme_classic())

personal_path = "/fh/working/sun_w/sshen/MorPhiC"
path = "/fh/fast/sun_w/MorPhiC/data/MorPhiC_exchange_experiment/"
dat_path = file.path(path, "DRACC-processed/MSK_exchange_experiment_DRACC_processed_March_2026/Tables")

metadata_path = file.path("/fh/working/sun_w/sshen/MorPhiC/")

read_header <- function(file_name, sheet){
  
  meta_header = fread(file_name)
  meta_header = as.character(meta_header)
  meta_header = gsub("_cell_line", "", meta_header, fixed = TRUE)
  meta_header = gsub("differentiated_product.", "", meta_header, fixed = TRUE)
  meta_header = gsub(".text", "", meta_header, fixed = TRUE)
  
  meta_header
}


Loading required package: viridisLite


Attaching package: ‘dplyr’


The following object is masked from ‘package:MASS’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
# Load required libraries
library(data.table)
library(stringr)
library(dplyr)

meta_file = paste0(metadata_path, "/MSK_meta_data.csv")

cl = fread(meta_file)
cl = as.data.frame(cl)[1:52, 1:12]
dim(cl)
meta_df = cl
colnames(meta_df) = meta_df[1, ]
meta_df = meta_df[-1, ]

# 2. Append the new formatted columns without removing existing ones
new_meta <- meta_df %>%
  mutate(
    Orig_ID = `CELL LINE ID SHORT NAME (Required)`,
    
    # Extract Replicate and Dcondition
    Replicate = str_extract(Orig_ID, "\\d+$"),
    Dcondition = str_replace(Orig_ID, "_\\d+$", ""),
    
    # Standard internal naming
    Name = paste0(Dcondition, "_Rep_", Replicate),
    
    Center = case_when(
      str_detect(`CELL LINE ID (Required)`, "NWU") ~ "NorthWestern",
      str_detect(`CELL LINE ID (Required)`, "UCSF") ~ "UCSF",
      str_detect(`CELL LINE ID (Required)`, "JAX") ~ "JAX",
      str_detect(`CELL LINE ID (Required)`, "MSK") ~ "MSK",
      str_detect(`CELL LINE ID (Required)`, "KOLF") ~ "Parent",
      TRUE ~ "Unknown"
    ),
    
    Treat = ifelse(`TREATMENT/CONDITION` == "not applicable" | is.na(`TREATMENT/CONDITION`), "none", `TREATMENT/CONDITION`),
    Condition = paste0(`CELL LINE DESCRIPTION`, " plus ", Treat),
    ko_gene = ifelse(str_detect(`CELL LINE DESCRIPTION`, "EOMES KO"), "KO", "WT")
  ) %>%
  select(-Orig_ID, -Treat)

# 3. RE-ASSIGN NAME TO MATCH cts COLUMNS
# This pastes the base condition and the replicate number together with just an underscore
new_meta$Name = paste0(new_meta$Dcondition, "_", new_meta$Replicate)
new_meta$Name = paste0(new_meta$Name, "_1_val")


[1] 52 12


## Check count data



In [3]:
cts = fread(file.path(dat_path, "genesCounts.csv"), data.table = FALSE)
cts = cts[, -1]
colnames(cts) <- gsub("_S[0-9]+_L[0-9]+$", "", colnames(cts))

new_meta$Name[new_meta$Name == "KOLF_3_1_val"] = "KOLF2_3_1_val"
new_meta = new_meta[match(colnames(cts), new_meta$Name), ]
fwrite(new_meta, file = file.path(personal_path, "MSK_meta_data.tsv"), sep="\t")



In [ ]:
gc()
sessionInfo()
